# LAB — Troubleshooting a Broken Job

**Goal:** diagnose and fix a deliberately broken 3-task Lakeflow Job using the run
history, the task DAG, error details, and Spark UI / query-profile metrics — the skills of
**exam domain 6 (Troubleshooting, Monitoring & Optimization)**.

**Scenario:** a teammate "refactored" RetailHub's nightly job on Friday. Since then
**`retailhub_broken_job`** (deployed by the trainer, sources in `materials/troubleshooting/broken_job/`)
is red every night: `ingest → transform → publish`. You have **view access** to the job — diagnose
from the run UI, and fix *scratch copies* here, writing `ts_*` tables into **your own catalog**.

**Prerequisites:**
- Trainer has deployed and run `retailhub_broken_job` (you have CAN VIEW); no CLI needed.

| Task | What you do | Where |
|------|-------------|-------|
| 1 | Read the failure from the run history & DAG, record the diagnosis | UI + notebook |
| 2 | Fix the ingest path + schema in a scratch copy and prove it runs | notebook |
| 3 | Diagnose a row-exploding join from query-profile metrics; fix the key | notebook |
| 4 | Right-size `spark.sql.shuffle.partitions` | notebook |
| 5 | Fix the broken dependency (publish reads a table nobody creates) | notebook |
| 6 | Bonus — cluster startup failures & library conflicts | notebook |

> ✅ **SOLUTION notebook** — full answers for `lab_troubleshooting`. Runs top-to-bottom on a fresh participant catalog (the broken job itself is never executed here — only fixed scratch copies).

## Setup

In [0]:
%run ../setup/00_setup

In [0]:
# Scratch tables for this lab — all in YOUR catalog, prefixed ts_
RAW_TABLE      = f"{CATALOG}.{BRONZE_SCHEMA}.ts_orders_raw"
ENRICHED_TABLE = f"{CATALOG}.{SILVER_SCHEMA}.ts_orders_enriched"
GOLD_TABLE     = f"{CATALOG}.{GOLD_SCHEMA}.ts_daily_sales"

ORDERS_JSON    = f"{DATASET_PATH}/orders/orders_batch.json"
CUSTOMERS_CSV  = f"{DATASET_PATH}/customers/customers.csv"

print(RAW_TABLE, ENRICHED_TABLE, GOLD_TABLE, sep="\n")

## Task 1 — Read the failure from the run history

Open **Jobs & Pipelines → `retailhub_broken_job` → latest run**. Look at the **DAG view** and the
**error details** of the red task, then fill in the diagnosis dict.

**What you need to do:**
1. In the DAG: which task is red? What state are the two downstream tasks in?
2. Click the red task → expand **Error** → read the **first line** of the exception.
3. Compare the failing path with the real Volume layout (`00_setup` printed it).

In [0]:
# Reproduce what the ingest task hit (read-only — safe to run).
# This is the exact path from materials/troubleshooting/broken_job/task_ingest.py:
BROKEN_PATH = f"/Volumes/{CATALOG}/default/dataset/orders/orders_batch.json"

try:
    spark.read.format("json").load(BROKEN_PATH).count()
    print("Unexpectedly worked?!")
except Exception as e:
    print(f"Error type : {type(e).__name__}")
    print(f"First line : {str(e).splitlines()[0][:160]}")
print(f"\nWorking base path from 00_setup: {DATASET_PATH}")

In [0]:
# Map what the UI shows to the allowed values:
#   - downstream tasks labelled "Upstream failed" in the DAG   -> "skipped" (they never ran)
#   - an error such as [UC_VOLUME_NOT_FOUND] or [PATH_NOT_FOUND] -> "path_not_found"
diagnosis_1 = {
    "failed_task":            "ingest",
    "downstream_task_state":  "skipped",   # shown as skipped / "Upstream failed"
    "error_category":         "path_not_found",
    "wrong_path_segment":     "dataset",   # the Volume is named "datasets"
}
print(diagnosis_1)

#### Hint — Task 1

**Run history → DAG → error, in that order.** The run list tells you *when* it started failing
(every run since Friday…), the DAG tells you *where* (first red node — everything after it is just
fallout), the error panel tells you *why*.

**Reading the DAG:** a task shown as *Upstream failed* (a skipped task) did not run at all — do not debug
it yet. Root-cause the **first** failure; downstream states are consequences.

**Reading the error:** the first line carries the error class, e.g.
`[UC_VOLUME_NOT_FOUND] Volume <catalog>.default.<name> does not exist` *(the path segment right after `default/` is parsed as the Volume name — a wrong folder inside an existing Volume would instead raise `[PATH_NOT_FOUND]`)*
— compare that path segment by segment with a path that works (`DATASET_PATH`).

In [0]:
# -- Validation --
assert diagnosis_1["failed_task"] == "ingest", "Which node is red FIRST in the DAG?"
assert diagnosis_1["downstream_task_state"] == "skipped", \
    "Tasks behind a failed dependency never run — check their state label"
assert diagnosis_1["error_category"] == "path_not_found", "Re-read the first line of the error"
assert str(diagnosis_1["wrong_path_segment"]).lower().strip("/") == "dataset", \
    "Compare the broken path with DATASET_PATH segment by segment"
print("Task 1 OK: root cause located in the ingest task (bad Volume path)")

## Task 2 — Fix path + schema in a scratch copy

The path is only fault #1. `task_ingest.py` also declares this reader schema:

```python
orders_schema = StructType([
    StructField("order_id",       StringType()),
    StructField("client_id",      StringType()),   # ???
    StructField("product_id",     StringType()),
    StructField("store_id",       StringType()),
    StructField("order_datetime", StringType()),
    StructField("quantity",       IntegerType()),
    StructField("unit_price",     DoubleType()),
    StructField("amount_total",   DoubleType()),   # ???
    StructField("payment_method", StringType()),
])
```

With a user-supplied schema, JSON fields that don't match a schema name are **silently NULL** —
so after fixing the path, the task dies on its quality gate: *"N/N rows have NULL
client_id/amount_total"*. The source file actually contains `customer_id` and `total_amount`.

**What you need to do:** write the corrected ingest (correct path via `ORDERS_JSON`, corrected
field names) into `RAW_TABLE`.

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType

fixed_schema = StructType([
    StructField("order_id",       StringType()),
    StructField("customer_id",    StringType()),   # was: client_id
    StructField("product_id",     StringType()),
    StructField("store_id",       StringType()),
    StructField("order_datetime", StringType()),
    StructField("quantity",       IntegerType()),
    StructField("unit_price",     DoubleType()),
    StructField("total_amount",   DoubleType()),   # was: amount_total
    StructField("payment_method", StringType()),
])

fixed_orders_df = (
    spark.read
    .format("json")
    .schema(fixed_schema)
    .load(ORDERS_JSON)
)

fixed_orders_df.write.mode("overwrite").saveAsTable(RAW_TABLE)
print(f"Written {RAW_TABLE}: {spark.table(RAW_TABLE).count():,} rows")

#### Hint — Task 2

**Why no error was thrown:** a JSON reader with an explicit schema does a *name-based* lookup per
field. A schema column that matches no JSON field is filled with NULL — no exception, no warning.
That is why quality gates (or `rescued data` in Auto Loader) matter at Bronze.

**The two wrong names:** peek at one raw line of the file to see the true field names:

```python
print(dbutils.fs.head(ORDERS_JSON, 300))
```

**Reader syntax** (the `write` line is already in the cell)

```python
fixed_orders_df = spark.read.format("<format>").schema(<schema>).load(<path>)
```

In [0]:
# -- Validation --
# This is the broken task's own quality gate — now it must pass
_raw   = spark.table(RAW_TABLE)
_total = _raw.count()
assert _total > 0, "RAW_TABLE is empty"
assert "customer_id" in _raw.columns and "total_amount" in _raw.columns, \
    f"Schema still wrong: {_raw.columns}"
_null_keys = _raw.filter("customer_id IS NULL OR total_amount IS NULL").count()
assert _null_keys < _total, "All keys NULL — a schema field name still doesn't match the JSON"
# ~3% of the source rows have a NULL customer_id / total_amount on purpose (dirty data), so allow up to 5%;
# a field name that still does not match the JSON makes that column NULL in 100% of rows.
assert _null_keys / _total < 0.05, f"{_null_keys}/{_total} NULL keys — check both corrected names"
print(f"Task 2 OK: quality gate passed — {_total - _null_keys:,}/{_total:,} rows populated")

## Task 3 — Diagnose the row-exploding join

With ingest fixed, `transform` "succeeds"… producing **~100× more rows than it reads**. The join
matches on `left(customer_id, 8)` — a truncated, **low-cardinality** key (`CUST0000`, `CUST0001`, …),
so every order matches ~100 customers.

**How this looks in the Spark UI / serverless query profile:**

| Metric (join stage) | Healthy enrichment join | This job |
|---|---|---|
| Output rows vs input rows | ≈ input (1:1) | **~100× input** |
| Shuffle write | small | inflated (all those duplicate rows) |
| Downstream write task time | seconds | minutes, all in the write after the join |

First **reproduce and measure**, then fill in the diagnosis, then **fix the join key**.

In [0]:
from pyspark.sql import functions as F

orders    = spark.table(RAW_TABLE)
customers = (spark.read.format("csv").option("header", True).load(CUSTOMERS_CSV))

orders_count = orders.count()

# The teammate's join, verbatim from task_transform.py:
exploded = orders.join(
    customers,
    F.substring(orders["customer_id"], 1, 8) == F.substring(customers["customer_id"], 1, 8),
    "inner",
)
exploded_count = exploded.count()

print(f"orders in    : {orders_count:>12,}")
print(f"joined out   : {exploded_count:>12,}   <- explosion factor ~{exploded_count/orders_count:.0f}x")
print("\nServerless: click 'See performance' under this cell's output → Query profile → join node → rows.")

In [0]:
distinct_bad_keys = (
    customers.select(F.substring("customer_id", 1, 8).alias("k")).distinct().count()
)

diagnosis_3 = {
    "symptom_metric":  "output_rows",
    "root_cause":      "truncated_low_cardinality_join_key",
    "fix":             "join_on_full_customer_id",
}
print(f"distinct truncated keys: {distinct_bad_keys}")
print(diagnosis_3)

In [0]:
enriched_fixed = orders.join(
    customers,
    orders["customer_id"] == customers["customer_id"],
    "inner",
).select(
    orders["*"],
    customers["first_name"],
    customers["last_name"],
    customers["city"],
    customers["country"],
    customers["customer_segment"],
)

enriched_fixed.write.mode("overwrite").saveAsTable(ENRICHED_TABLE)
print(f"{ENRICHED_TABLE}: {spark.table(ENRICHED_TABLE).count():,} rows")

#### Hint — Task 3

**Where to look on classic compute:** Spark UI → **SQL/DataFrame** tab → the join query → check
*output rows* of the SortMergeJoin/ShuffleHashJoin node vs its inputs; then **Stages** for shuffle
read/write sizes. On **serverless**: click **See performance** under the cell output → **Query profile** →
join node → rows — same numbers, different door.

**Confirm cardinality with data**, not intuition — count distinct values of the actual join
expression: `F.substring(col, 1, 8)` builds the truncated key, `.select(...).distinct().count()` counts its values.

**The fix** is never "more compute" — join on the real key (`customer_id`, full length), which is
unique per customer, so an inner join returns at most one customer per order.

In [0]:
# -- Validation --
assert 50 <= distinct_bad_keys <= 200, \
    f"Expected ~100 distinct truncated keys, got {distinct_bad_keys} — count left(customer_id, 8) over customers"
assert diagnosis_3["symptom_metric"] == "output_rows", "The smoking gun is the join node's output rows"
assert diagnosis_3["root_cause"] == "truncated_low_cardinality_join_key", "Re-check the join condition"
assert diagnosis_3["fix"] == "join_on_full_customer_id", "The fix is the key, not the cluster"

_enriched_count = spark.table(ENRICHED_TABLE).count()
assert _enriched_count < exploded_count / 10, \
    f"Still exploding: {_enriched_count:,} rows — is the join on the FULL customer_id?"
assert _enriched_count <= orders_count, "An enrichment inner join must not multiply orders"
assert _enriched_count >= orders_count * 0.9, \
    f"Lost too many rows ({_enriched_count:,} of {orders_count:,}) — wrong join type or key?"
print(f"Task 3 OK: {_enriched_count:,} rows (was {exploded_count:,} exploded)")

## Task 4 — Right-size the shuffle partitions

`task_transform.py` also sets:

```python
spark.conf.set("spark.sql.shuffle.partitions", "4000")
```

…copied from a blog post about a 50 TB workload. Our shuffle here is a few hundred MB — 4000
partitions means thousands of **tiny tasks** where scheduling overhead dwarfs the work, and
(on classic compute) 4000 tiny output files per shuffle.

**What you need to do:** observe the effect, then set a sensible value and fill in the answer dict.

In [0]:
import time

def timed_groupby():
    t0 = time.time()
    (spark.table(ENRICHED_TABLE)
        .groupBy("country", "payment_method")
        .count()
        .collect())
    return time.time() - t0

spark.conf.set("spark.sql.shuffle.partitions", "4000")   # the teammate's setting
t_bad = timed_groupby()
print(f"shuffle.partitions=4000 : {t_bad:.1f}s")

In [0]:
spark.conf.set("spark.sql.shuffle.partitions", "auto")

t_good = timed_groupby()
print(f"shuffle.partitions={spark.conf.get('spark.sql.shuffle.partitions')} : {t_good:.1f}s (was {t_bad:.1f}s)")

answer_4 = {
    "why_4000_is_bad": "tiny_task_scheduling_overhead",
    "better_value":    "auto",
}
print(answer_4)

#### Hint — Task 4

**How to spot it in the Spark UI:** Stages tab → the shuffle stage shows a huge **task count**
(= `spark.sql.shuffle.partitions`) with median task duration of a few *milliseconds* — a classic
"death by overhead" pattern. Compare *task count × median time* with the stage wall-clock.

**Right-sizing rules of thumb**
- On **serverless** (and any AQE-enabled DBR): leave it on **`"auto"`** — AQE coalesces post-shuffle
  partitions to sane sizes; that is also the serverless default.
- Manual tuning target (classic clusters): ~**128–200 MB per shuffle partition**, i.e.
  `shuffle data size / 128MB` — for a few hundred MB that is single-digit-to-tens, nowhere near 4000.

`spark.sql.shuffle.partitions` is one of the few Spark confs you may still set on serverless.

In [0]:
# -- Validation --
assert answer_4["why_4000_is_bad"] == "tiny_task_scheduling_overhead", \
    "4000 partitions over ~hundreds of MB = thousands of ms-long tasks"
_bv = answer_4["better_value"]
assert _bv == "auto" or (isinstance(_bv, int) and 1 <= _bv <= 200), \
    "Use 'auto' (AQE/serverless) or a small integer sized at ~128MB per partition"
_current = str(spark.conf.get("spark.sql.shuffle.partitions"))
assert _current != "4000", "The session still runs with 4000 — set your better_value for real"
print(f"Task 4 OK: shuffle.partitions={_current}")

## Task 5 — Fix the broken dependency

The last task, `publish`, has **two** wiring problems:

**1. It reads a table nobody creates.** From `task_publish.py`:

```python
SOURCE = f"{CATALOG}.silver.ts_orders_final"   # transform writes ts_orders_enriched!
```

**2. Wrong DAG edge.** In the job definition, `publish` depends only on `ingest` — so even with a
correct table name it could run **before/parallel to** `transform` (race condition):

```json
{ "task_key": "publish", "depends_on": [ { "task_key": "ingest" } ] }
```

**What you need to do:** reproduce the error, fill in the diagnosis, then run a corrected publish
into `GOLD_TABLE`.

> You can't edit the job (view-only) — record the fix; the trainer applies it.

In [0]:
# Reproduce the publish failure (read-only)
try:
    spark.table(f"{CATALOG}.{SILVER_SCHEMA}.ts_orders_final").count()
except Exception as e:
    print(f"Error type : {type(e).__name__}")
    print(f"First line : {str(e).splitlines()[0][:160]}")

print("\nTables that DO exist in silver:")
display(spark.sql(f"SHOW TABLES IN {CATALOG}.{SILVER_SCHEMA} LIKE 'ts_*'"))

In [0]:
diagnosis_5 = {
    "missing_table":           "ts_orders_final",
    "table_actually_produced": "ts_orders_enriched",
    "dag_fix":                 "publish_depends_on_transform",
}
print(diagnosis_5)

from pyspark.sql import functions as F
daily_sales_fixed = (
    spark.table(ENRICHED_TABLE)
    .groupBy(F.to_date("order_datetime").alias("order_date"), "country")
    .agg(
        F.count("*").alias("order_count"),
        F.round(F.sum("total_amount"), 2).alias("revenue"),
        F.round(F.avg("total_amount"), 2).alias("avg_order_value"),
    )
)
daily_sales_fixed.write.mode("overwrite").saveAsTable(GOLD_TABLE)
print(f"{GOLD_TABLE}: {spark.table(GOLD_TABLE).count():,} rows")

#### Hint — Task 5

**`TABLE_OR_VIEW_NOT_FOUND` in a multi-task job** almost always means one of:
1. the producing task writes a **different name** than the consumer reads (rename drift — our case),
2. the consumer runs **before** the producer (missing `depends_on` edge — also our case),
3. wrong catalog/schema context.

**Fix both ends:** point `SOURCE` at what transform really writes, *and* add
`depends_on: [transform]` to publish in the job definition — a correct name with a wrong edge is a
flaky job that fails only when publish wins the race.

**The corrected aggregation** (as in `task_publish.py`, with the right source `ENRICHED_TABLE`) produces:

| Output column | Expression |
|---|---|
| `order_date` | group key: `F.to_date("order_datetime")` |
| `country` | group key |
| `order_count` | `F.count("*")` |
| `revenue` | `F.round(F.sum("total_amount"), 2)` |
| `avg_order_value` | `F.round(F.avg("total_amount"), 2)` |

In [0]:
# -- Validation --
assert str(diagnosis_5["missing_table"]).lower() == "ts_orders_final"
assert str(diagnosis_5["table_actually_produced"]).lower() == "ts_orders_enriched"
assert diagnosis_5["dag_fix"] == "publish_depends_on_transform", \
    "publish consumes transform's output — the DAG edge must say so"

_gold = spark.table(GOLD_TABLE)
assert _gold.count() > 0, "GOLD_TABLE is empty"
for _c in ["order_date", "country", "order_count", "revenue", "avg_order_value"]:
    assert _c in _gold.columns, f"Missing column {_c} in {GOLD_TABLE}: {_gold.columns}"
print(f"Task 5 OK: publish fixed — {_gold.count():,} daily rows in {GOLD_TABLE}")

## Task 6 — Bonus: Failures that happen *before* your code runs

Two scenarios that show up on the exam but can't be reproduced in a notebook:

**Scenario A — cluster startup failure.** Monday 06:00, the nightly job (classic cluster) fails in
0 seconds of task time: *"Cluster terminated. Reason: Init script failure"*. No notebook cell ever
ran.
- Where do you look first? (Hint: not the notebook, not the job run's Spark UI.)
- Typical causes: init script non-zero exit, instance capacity, driver OOM at start.

**Scenario B — library conflict.** After someone added `pandas==2.2` as a *cluster* library, an
unrelated job on the same cluster dies at import with
`AttributeError`/`ClassNotFoundException`-style errors.
- What's the cleanest way to give ONE job its own library versions without touching the cluster?

Fill in the quick answers:

In [0]:
bonus_6 = {
    "startup_failure_first_look": "cluster_event_log",   # Compute -> cluster -> Event log (+ init script log)
    "library_conflict_fix":       "notebook_scoped_pip", # %pip install pandas==2.2 in THAT job's notebook, pinned
}
print(bonus_6)

In [0]:
# -- Validation --
assert bonus_6["startup_failure_first_look"] == "cluster_event_log", \
    "Startup failures happen before user code — Compute -> Event log (init script log)"
assert bonus_6["library_conflict_fix"] == "notebook_scoped_pip", \
    "%pip installs are scoped to one session — no cluster-wide blast radius"
print("Task 6 OK")

## Summary — mapping to exam domain 6 (Troubleshooting, Monitoring & Optimization)

| Task | What you practiced | Domain 6 objective |
|------|--------------------|--------------------|
| 1 | Run history → DAG → first red task → error first line | Diagnose job failures from run details; distinguish failed vs skipped (upstream-failed) tasks |
| 2 | Silent NULLs from a mismatched user schema; quality gates | Root-cause schema mismatches & bad paths; why explicit-schema reads fail *quietly* |
| 3 | Join-node output rows, shuffle write, cardinality checks | Use Spark UI / query profile stage metrics to find row explosion & skew-like symptoms |
| 4 | `spark.sql.shuffle.partitions`, AQE `auto`, task-count overhead | Right-size shuffle partitions; recognize tiny-task overhead patterns |
| 5 | Producer/consumer name drift + missing `depends_on` edge | Fix task dependency/ordering bugs; `TABLE_OR_VIEW_NOT_FOUND` triage in multi-task jobs |
| 6 | Cluster event log; notebook-scoped `%pip` | Diagnose cluster startup failures & library conflicts |

**The method, portable to any broken job:** run history (when) → DAG (where) → error first line
(what) → stage metrics (why slow/exploding) → fix the *first* failure and re-run downstream
(Repair run re-executes only failed + downstream tasks).

## Cleanup

In [0]:
for _t in [RAW_TABLE, ENRICHED_TABLE, GOLD_TABLE]:
    spark.sql(f"DROP TABLE IF EXISTS {_t}")
print("Lab cleanup complete")

← [Lab — Troubleshooting](../day3/lab/lab_troubleshooting.ipynb) | **[README](../../README.md)**